# 21. PCA 차원 축소와 군집·분류 비교 (PCA Dimensionality Reduction)

고차원(50차원) 데이터에서 PCA 2차원 축소 전후의 KMeans 실루엣 스코어를 비교하고, Iris 데이터에서 데이터 누수 없는 `Pipeline(StandardScaler + PCA + KNN)` 분류 성능을 확인합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# 1. 고차원(50D) 데이터 생성 및 스케일링
X_hd, _ = make_classification(
    n_samples=300, n_features=50, n_informative=5, n_redundant=0, n_clusters_per_class=1, random_state=42
)
X_scaled = StandardScaler().fit_transform(X_hd)

# PCA로 50D -> 2D 축소
pca_2d = PCA(n_components=2, random_state=42)
X_pca = pca_2d.fit_transform(X_scaled)

# 원본 50D vs PCA 2D KMeans 실루엣 스코어 비교
labels_hd = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_scaled)
labels_pca = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(X_pca)

print(f'Silhouette (50D):    {silhouette_score(X_scaled, labels_hd):.4f}')
print(f'Silhouette (PCA-2D): {silhouette_score(X_pca, labels_pca):.4f}')
print(f'설명 분산 비율 합계: {pca_2d.explained_variance_ratio_.sum():.4f}')

In [ ]:
# 동일한 2D PCA 평면 위에서 두 군집 결과 시각화
fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
ax[0].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_hd, cmap='viridis', s=35)
ax[0].set_title('KMeans on 50D')
ax[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_pca, cmap='viridis', s=35)
ax[1].set_title('KMeans on PCA-2D')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Iris 데이터: 데이터 누수 방지를 위해 분할 후 Pipeline 적용
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(f'Iris PCA-2D + KNN Test Accuracy: {accuracy_score(y_test, pred):.4f}')